# 01 · Recolección de datos (scraping)

**Proyecto:** mercado de motos, bicicletas y patinetas en Cuba  
**Fuente:** Revolico (anuncios públicos)  
**Objetivo:** descargar los anuncios de las categorías del estudio y
guardarlos en `data/processed/anuncios.csv`, conservando el HTML crudo
en `data/raw/`.

> Antes de ejecutar: revisa `docs/metodologia.md` y asegúrate de tener
> `requests` instalado. El scraper espera 2–4 s entre páginas; no lo
> acelere.


## 0. Preparación del entorno

Añadimos la raíz del proyecto a `sys.path` para poder importar `src`.

In [ ]:
# Permite importar el paquete src/ desde la raíz del proyecto.
import sys
from pathlib import Path

RAIZ = Path.cwd()
if not (RAIZ / 'src').exists():
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

print('Raíz del proyecto:', RAIZ)

## 1. ¿De dónde salen los datos?

Revolico renderiza en el servidor e incrusta un JSON en
`<script id="__NEXT_DATA__">`. Dentro está el estado de Apollo, con los
anuncios como claves `AdType:<id>`.

Veamos la configuración de categorías del proyecto.

In [ ]:
from src.config import BASE_URL, CATEGORIAS, COLUMNAS

print('Base:', BASE_URL)
for nombre, params in CATEGORIAS.items():
    print(f'- {nombre}: {params}')
print('\nColumnas del CSV:', COLUMNAS)

## 2. Inspección de una página antes de scrapear

Descargamos una sola página y miramos la estructura del JSON. Esto sirve
para detectar si el sitio cambió.

In [ ]:
import json
from urllib.parse import urlencode

import requests

from src.config import HEADERS
from src.scraper import parse_state

params = {**CATEGORIAS['bicicletas'], 'cu': '1', 'page': 1}
url = f'{BASE_URL}/search?{urlencode(params)}'
print('URL:', url)

respuesta = requests.get(url, headers=HEADERS, timeout=30)
respuesta.raise_for_status()
html = respuesta.text
print('HTML descargado:', len(html), 'caracteres')

state = parse_state(html)
anuncios = {k: v for k, v in state.items() if k.startswith('AdType:')}
print('Anuncios en la página:', len(anuncios))
print('\nPrimer anuncio:')
print(json.dumps(next(iter(anuncios.values())), ensure_ascii=False, indent=2)[:1200])

Observa que el anuncio **no** trae descripción ni teléfono en el listado.
La descripción requiere visitar la página de detalle (ver
`src/details.py`).

## 3. Extracción de una página a tabla

In [ ]:
from src.scraper import extraer_anuncios

filas = extraer_anuncios(state, categoria='bicicletas', fecha_captura='demo')
print('Filas:', len(filas))
for fila in filas[:3]:
    print(fila)

## 4. Prueba controlada: una categoría, una página

Ejecuta esto primero para validar que todo funciona. Guarda el HTML en
`data/raw/` y devuelve las filas en memoria (no escribe CSV todavía).

In [ ]:
from src.scraper import scrape_category

filas_prueba = scrape_category(
    categoria='bicicletas',
    params=CATEGORIAS['bicicletas'],
    max_pages=1,
)
print('Total:', len(filas_prueba))

## 5. Recolección completa (desde la terminal)

Para recolectar todas las categorías, ejecuta en una terminal (no aquí,
para no bloquear el notebook):

```bash
python -m src.scraper --max-pages 43
```

Si quieres hacerlo desde el notebook, descomenta y ajusta las páginas.
**Advertencia:** con 43 páginas y 2–4 s de pausa, tarda decenas de minutos.

In [ ]:
# import subprocess
# subprocess.run([sys.executable, '-m', 'src.scraper', '--max-pages', '43'], check=True)

## 6. Verificación del CSV generado

Si ya ejecutaste el scraper, comprobamos el archivo.

In [ ]:
import pandas as pd
from src.config import DATA_PROCESSED

ruta = DATA_PROCESSED / 'anuncios.csv'
if ruta.exists():
    df = pd.read_csv(ruta)
    print('Filas:', len(df))
    print('Columnas:', list(df.columns))
    print('\nAnuncios por categoría:')
    print(df['categoria'].value_counts().to_string())
    display(df.head())
else:
    print('Aún no existe', ruta, '- ejecuta el scraper primero.')

## 7. Registro de captura

Anota la fecha, el comando y el número de anuncios en
`docs/metodologia.md` (sección *Registro de captura*). Es parte de la
reproducibilidad del proyecto.

### Notas de ética

- `robots.txt` permite el acceso salvo checkout, cuenta, auth y favoritos.
- Pausa de 2–4 s, sin paralelismo, `User-Agent` identificable.
- No se guardan teléfonos, nombres ni IDs de usuario.
- Uso exclusivamente académico.
